# Dys model

### Imports

In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import cumulative_dynamic_auc

### Model classes

In [3]:
class SmoothStepGate(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.mu = nn.Parameter(torch.randn(size) * 0.01) 
    def forward(self):
        return torch.clamp(self.mu + 0.5, 0, 1)

class ShapeFunction(nn.Module):
    def __init__(self, K):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, K)
        )
    def forward(self, x): 
        return self.net(x)

class DySModel(nn.Module):
    def __init__(self, num_features, K):
        super().__init__()
        self.K = K
        self.num_features = num_features
        self.main_effects = nn.ModuleList([ShapeFunction(K) for _ in range(num_features)])
        self.main_gates = SmoothStepGate(num_features)
        self.intercept = nn.Parameter(torch.zeros(1, K))

    def forward(self, x):
        s_m = self.main_gates()
        logits = self.intercept.expand(x.shape[0], -1).clone()
        for i in range(self.num_features):
            logits += self.main_effects[i](x[:, i:i+1]) * s_m[i]
        pmf = torch.softmax(logits, dim=1)
        return 1 - torch.cumsum(pmf, dim=1)

## loss and evaluation

In [ ]:
def omega(x):
    x = torch.clamp(x, 1e-7, 1-1e-7)
    return -(x * torch.log(x) + (1 - x) * torch.log(1 - x))

def compute_dys_loss(surv, t, e, eval_times, model, l_lambda, tau=0.1):
    t_eval = torch.tensor(eval_times, device=surv.device).float().unsqueeze(0)
    t_actual = t.unsqueeze(1)
    mask_before = (t_eval < t_actual).float()
    mask_after = (t_eval >= t_actual).float()
    l_rps = ((1 - surv).pow(2) * mask_before).sum(dim=1) + \
            (surv.pow(2) * mask_after * e.unsqueeze(1)).sum(dim=1)
    s_m = model.main_gates()
    return l_rps.mean() + (l_lambda * s_m.sum()) + (tau * omega(s_m).sum())

def evaluate_auc(model, x_test, t_train, e_train, t_test, e_test, eval_times):
    model.eval()
    with torch.no_grad():
        x_te = torch.tensor(x_test, dtype=torch.float32).to(device)
        surv_pred = model(x_te).cpu().numpy()
        risk_scores = 1 - surv_pred[:, len(eval_times)//2]
        y_train = np.array([(bool(e), t) for e, t in zip(e_train, t_train)], dtype=[('event', 'bool'), ('time', 'float')])
        y_test = np.array([(bool(e), t) for e, t in zip(e_test, t_test)], dtype=[('event', 'bool'), ('time', 'float')])
        times = np.percentile(t_test[e_test == 1], [25, 50, 75])
        auc, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk_scores, times)
        return mean_auc

### Plotting functions

In [ ]:
def plot_impact_curves(model, x_train_scaled, feature_names, eval_times, scaler, feature_to_plot):
    """
    Побудова графіків впливу (Impact Plots) згідно з Розділом 3.1 та Додатком A.2
    """
    model.eval()
    if feature_to_plot not in feature_names:
        print(f"Ознака {feature_to_plot} не знайдена.")
        return

    idx = feature_names.index(feature_to_plot)
    
    # 1. Створюємо діапазон значень для осі X (від min до max даної ознаки)
    x_min, x_max = x_train_scaled[:, idx].min(), x_train_scaled[:, idx].max()
    x_range_scaled = np.linspace(x_min, x_max, 100).reshape(-1, 1)
    x_range_tensor = torch.tensor(x_range_scaled, dtype=torch.float32).to(device)
    
    # 2. Отримуємо "сирий" вплив від Shape Function
    with torch.no_grad():
        impact = model.main_effects[idx](x_range_tensor).cpu().numpy()
        
        # Центрування (згідно з A.2): віднімаємо середній вихід на тренувальних даних
        train_feat_tensor = torch.tensor(x_train_scaled[:, idx:idx+1], dtype=torch.float32).to(device)
        mean_offset = model.main_effects[idx](train_feat_tensor).mean(dim=0).cpu().numpy()
        impact = impact - mean_offset

    # 3. Денормалізація осі X для зрозумілості (повертаємо реальні одиниці виміру)
    x_actual = x_range_scaled.flatten() * np.sqrt(scaler.var_[idx]) + scaler.mean_[idx]

    # 4. Вибираємо часові точки для порівняння (наприклад, 1 рік та 5 років)
    # Припустимо, час у днях. 365 днів та 1825 днів.
    t1_idx = np.abs(eval_times - 365).argmin()
    t5_idx = np.abs(eval_times - 1825).argmin()

    plt.figure(figsize=(10, 6))
    plt.plot(x_actual, impact[:, t1_idx], label=f'Вплив на t={int(eval_times[t1_idx])} (1 рік)', color='blue')
    plt.plot(x_actual, impact[:, t5_idx], label=f'Вплив на t={int(eval_times[t5_idx])} (5 років)', color='red', linestyle='--')
    
    plt.axhline(0, color='black', lw=1, ls='-')
    plt.xlabel(f"Значення ознаки: {feature_to_plot}")
    plt.ylabel("Внесок у логіт (Centered Impact)")
    plt.title(f"Impact Plot: {feature_to_plot} (DyS Glass-Box)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

### Bisection